Определение орбиты методом Лапласа (тест из эфемерид)

In [21]:
from math import *
import numpy as np

import pandas as pd

import os
import subprocess
from concurrent.futures import ThreadPoolExecutor

from astropy.time import Time

In [22]:
ASTEPH_DIR = '/home/pavel/Programs/astEph' # локальная программа astEph с MPCORB.DAT
ASTEPH_RATE_STEP = 300.0    # с - шаг численного дифференцирования для скоростей из astEph
ASTID = '24445'
MU = 2.9591220828411956E-04 # а.е.^3/сут^2 - GM Солнца (как _GMS в astEph, = k^2)
C_AU_DAY = 299792.458*86400/1.495978707E+08 # скорость света, а.е./сут
EPS_J2000 = np.radians(23.43929111)         # наклон эклиптики J2000 (как в astEph)



N = 10
t = Time('2026-09-29T15:35:23.153', format='isot')
JD = float(t.copy(format='jd').value)+5*np.arange(10)/1440.0
JD

array([2461313.14957353, 2461313.15304575, 2461313.15651797,
       2461313.1599902 , 2461313.16346242, 2461313.16693464,
       2461313.17040686, 2461313.17387909, 2461313.17735131,
       2461313.18082353])

In [23]:
def _run_asteph(target, jd, location):
    """Один вызов astEph: (RA, DEC в градусах, V) на момент jd (UTC) для обсерватории location."""
    exe = os.path.join(ASTEPH_DIR, 'astEph')
    out = subprocess.run([exe, str(target), f'{jd:.8f}', f'obscode={location}'], cwd=ASTEPH_DIR,
                         capture_output=True, text=True, timeout=120)
    if out.returncode != 0:
        raise RuntimeError(f'astEph: {out.stderr.strip() or out.stdout.strip()}')
    val = {l.split('=')[0].strip(): l.split('=', 1)[1].split()[0] for l in out.stdout.splitlines() if '=' in l}
    return np.degrees(float(val['RA'])), np.degrees(float(val['DEC'])), float(val['Vmag'])

def get_asteph_ephemeris(target, JD, location):
    """Та же эфемерида, что get_horizons_ephemeris, но из локальной программы astEph (ASTEPH_DIR).
    astEph дает только положение и V, поэтому скорости считаются по положениям в моменты
    jd -+ ASTEPH_RATE_STEP (с), а DELTA не заполняется. Первый вызов делается отдельно: при новой
    эпохе MPCORB.DAT astEph сначала считает и сохраняет свой кэш интегрирования (~10 с). Остальные
    вызовы идут параллельно."""
    JD = np.atleast_1d(np.asarray(JD, dtype=float))
    if not os.path.isfile(os.path.join(ASTEPH_DIR, 'astEph')):
        raise FileNotFoundError(f'нет программы {ASTEPH_DIR}/astEph (сборка: cc -O2 -o astEph astEph.c -lm)')
    h = ASTEPH_RATE_STEP/86400
    jds = np.concatenate([JD, JD-h, JD+h])
    first = _run_asteph(target, jds[0], location)
    with ThreadPoolExecutor(max_workers=8) as ex:
        rest = list(ex.map(lambda jd: _run_asteph(target, jd, location), jds[1:]))
    res = np.array([first] + rest).reshape(3, len(JD), 3)
    (ra, dec, v), (ra1, dec1, _), (ra2, dec2, _) = [res[k].T for k in range(3)]
    dra = ((ra2-ra1+180) % 360 - 180)*np.cos(np.radians(dec))*3600/(2*ASTEPH_RATE_STEP/3600)
    ddec = (dec2-dec1)*3600/(2*ASTEPH_RATE_STEP/3600)
    return pd.DataFrame(dict(JD=JD, RA_EPH=ra, DEC_EPH=dec, dRAcosD=dra, dDEC=ddec, # arcsec/hour
                             VMAG=v, DELTA=np.nan))


def get_observer_state(JD, location='217', asteph_dir=ASTEPH_DIR):
    """Гелиоцентрические положение R (а.е.), скорость V (а.е./сут) и ускорение A (а.е./сут^2)
    наблюдателя в обсерватории с кодом MPC location на момент JD (UTC), экваториальная система J2000.
    Считает программа astEph (режим --observer). Возвращает три numpy-массива по 3 компоненты."""
    out = subprocess.run([os.path.join(asteph_dir, 'astEph'), '--observer', f'{float(JD):.8f}',
                          f'obscode={location}'], cwd=asteph_dir, capture_output=True, text=True, timeout=300)
    if out.returncode != 0:
        raise RuntimeError(f'astEph: {out.stderr.strip() or out.stdout.strip()}')
    val = {k.strip(): v for k, v in (l.split('=', 1) for l in out.stdout.splitlines() if '=' in l)}
    R, V, A = (np.array(val[k].split(), dtype=float) for k in ('R', 'V', 'A'))
    return R, V, A

In [24]:
def analyze_vector_D(pos, n):
    # Вычисляем JD0
    jd0 = pos['JD'].mean()

    # Инициализируем массивы
    D = np.zeros(3)
    dotD = np.zeros(3)
    ddotD = np.zeros(3)

    # Вычисляем компоненты D
    pos['D0'] = np.cos(np.radians(pos['RA_EPH'])) * np.cos(np.radians(pos['DEC_EPH']))
    pos['D1'] = np.sin(np.radians(pos['RA_EPH'])) * np.cos(np.radians(pos['DEC_EPH']))
    pos['D2'] = np.sin(np.radians(pos['DEC_EPH']))

    # Инициализируем результаты
    results = []

    for j in range(3):
        DD = pos[f'D{j}']
        JD = pos['JD']

        # Вычисляем x = JD - JD0
        x = JD - jd0

        # Построение модели D = sum C_k * x^k
        X = np.column_stack([x**k for k in range(n+1)])

        # Метод наименьших квадратов
        C, residuals, rank, singular_values = np.linalg.lstsq(X, DD, rcond=None)

        D[j] = C[0]

        # Вычисляем первые и вторые производные в точке x=0
        if n >= 1:
            dotD[j] = C[1]
        else:
            dotD[j] = 0
        if n >= 2:
            ddotD[j] = 2 * C[2]
        else:
            ddotD[j] = 0
            
    return jd0, D, dotD, ddotD

In [25]:
pos = get_asteph_ephemeris(ASTID, JD, '217')
pos

,JD,RA_EPH,DEC_EPH,dRAcosD,dDEC,VMAG,DELTA
0,2.461313e+06,296.515013,11.887703,11.205227,-72.086603,16.49,NaN
1,2.461313e+06,296.515278,11.886034,11.232201,-72.097165,16.49,NaN
2,2.461313e+06,296.515544,11.884365,11.260158,-72.107741,16.49,NaN
3,2.461313e+06,296.515811,11.882696,11.289054,-72.118123,16.49,NaN
4,2.461313e+06,296.516078,11.881026,11.318878,-72.128297,16.49,NaN
5,2.461313e+06,296.516346,11.879357,11.349652,-72.138462,16.49,NaN
6,2.461313e+06,296.516615,11.877687,11.381366,-72.148628,16.49,NaN
7,2.461313e+06,296.516885,11.876016,11.413974,-72.158574,16.49,NaN
8,2.461313e+06,296.517155,11.874346,11.447465,-72.168295,16.49,NaN
9,2.461313e+06,296.517426,11.872675,11.481860,-72.178006,16.49,NaN


In [26]:
JD0, D, dotD, ddotD = analyze_vector_D(pos, n=3)
D

array([ 0.43688814, -0.87564148,  0.20586588])

In [27]:
g, dot_g, ddot_g = get_observer_state(JD0, location='217', asteph_dir=ASTEPH_DIR)
g, dot_g, ddot_g

(array([0.99594586, 0.09856571, 0.04275358]),
 array([-0.00200814,  0.01579159,  0.00677491]),
 array([-1.30938116e-03,  6.73224227e-04, -9.29615479e-06]))

In [28]:
def laplace_coefficients(D, dotD, ddotD, g, ddot_g):
    """Коэффициенты уравнений C d = C2 + C3 r^-3 и r^2 = d^2 + 2 C1 d + C0."""
    DxdD = np.cross(D, dotD)
    return dict(C=ddotD @ DxdD, C0=g @ g, C1=g @ D, C2=-ddot_g @ DxdD, C3=-MU*(g @ DxdD))

coef = laplace_coefficients(D, dotD, ddotD, g, ddot_g)
pd.Series(coef)

C     0.000008
C0    1.003451
C1    0.357610
C2    0.000007
C3   -0.000002
dtype: float64

In [29]:
D_MIN = 1e-4     # а.е. (~15000 км) - решения ближе к наблюдателю считаются ложными
ROOT_INDEX = 0   # какое из оставшихся решений брать, если их несколько

def solve_rd(C, C0, C1, C2, C3):
    """Все положительные вещественные корни уравнения Лагранжа и соответствующие d (из lagrange_rd.tex)."""
    al, be = C2 / C, C3 / C
    a = -(al**2 + 2*C1*al + C0)
    b = -2*be*(al + C1)
    c = -be**2
    roots = np.roots([1, 0, a, 0, 0, b, 0, 0, c]) # коэффициенты при r^8, r^7, ..., r^0
    sols = []
    for r in roots:
        if abs(r.imag) < 1e-10 and r.real > 0:
            r = r.real
            sols.append((r, al + be / r**3))
    return sorted(sols)

sols = solve_rd(**coef)
for r_, d_ in sols:
    print(f'r = {r_:.6f} а.е., d = {d_:+.6f} а.е.' + ('' if d_ > D_MIN else '  <- отброшено (d <= D_MIN)'))
good = [(r_, d_) for r_, d_ in sols if d_ > D_MIN]
if not good:
    raise RuntimeError('нет физического решения (d > D_MIN)')
r, d = good[min(ROOT_INDEX, len(good)-1)]
print(f'выбрано: r = {r:.6f} а.е., d = {d:.6f} а.е.')

r = 1.458301 а.е., d = +0.760905 а.е.
выбрано: r = 1.458301 а.е., d = 0.760905 а.е.


In [30]:
def laplace_state(D, dotD, ddotD, g, dot_g, ddot_g, r, d, coef):
    """dd/dt и гелиоцентрические векторы положения и скорости астероида."""
    DxddD = np.cross(D, ddotD)
    dot_d = (ddot_g @ DxddD + MU/r**3*(g @ DxddD))/(2*coef['C'])
    return dot_d, g + d*D, dot_g + dot_d*D + d*dotD

dot_d, r_vec, v_vec = laplace_state(D, dotD, ddotD, g, dot_g, ddot_g, r, d, coef)
JD_EPOCH = JD0 - d/C_AU_DAY
print(f'dd/dt = {dot_d:+.6f} а.е./сут, световое запаздывание {d/C_AU_DAY*86400:.1f} с, эпоха JD {JD_EPOCH:.6f} UTC')
print('r =', r_vec, f' а.е. (|r| = {np.linalg.norm(r_vec):.6f}, из уравнения Лагранжа r = {r:.6f})')
print('v =', v_vec, ' а.е./сут')

dd/dt = -0.001586 а.е./сут, световое запаздывание 379.7 с, эпоха JD 2461313.160804 UTC
r = [ 1.32837631 -0.56771444  0.19939799]  а.е. (|r| = 1.458301, из уравнения Лагранжа r = 1.458301)
v = [-0.00121623  0.01645203  0.00019881]  а.е./сут


In [31]:
def orbital_elements(r_eq, v_eq):
    """Гелиоцентрические элементы (эклиптика и равноденствие J2000) по векторам в экваториальной системе.
    a в а.е., углы в градусах; для гиперболы M = nan."""
    ce, se = np.cos(EPS_J2000), np.sin(EPS_J2000)
    Rot = np.array([[1, 0, 0], [0, ce, se], [0, -se, ce]])
    rv, vv = Rot @ r_eq, Rot @ v_eq
    rn = np.linalg.norm(rv)
    h = np.cross(rv, vv)
    a = 1/(2/rn - vv @ vv/MU)
    ev = np.cross(vv, h)/MU - rv/rn
    e = np.linalg.norm(ev)
    inc = degrees(acos(h[2]/np.linalg.norm(h)))
    node = np.cross([0, 0, 1], h)
    Om = degrees(atan2(node[1], node[0])) % 360
    w = degrees(acos(np.clip(node @ ev/np.linalg.norm(node)/e, -1, 1)))
    w = w if ev[2] >= 0 else 360 - w
    nu = acos(np.clip(ev @ rv/e/rn, -1, 1))
    nu = nu if rv @ vv >= 0 else 2*pi - nu
    if e < 1:
        E = 2*atan(sqrt((1-e)/(1+e))*tan(nu/2))
        M = degrees(E - e*sin(E)) % 360
    else:
        M = np.nan
    return dict(a=a, e=e, i=inc, Node=Om, Peri=w, M=M)

def _unpack_mpc_epoch(s):
    """Упакованная эпоха MPC (напр. K2669) -> JD (TT, 0h)."""
    dig = lambda c: int(c) if c.isdigit() else ord(c) - ord('A') + 10
    year = (ord(s[0]) - ord('A') + 10)*100 + int(s[1:3])
    return Time(f'{year:04d}-{dig(s[3]):02d}-{dig(s[4]):02d}', scale='tt').jd

def read_mpcorb(target, path=os.path.join(ASTEPH_DIR, 'MPCORB.DAT')):
    """Элементы астероида target из MPCORB.DAT (номер ищется как '(номер)' в читаемом обозначении)."""
    key = str(target).strip()
    with open(path, errors='replace') as f:
        for line in f:
            name = line[166:194].strip() if len(line) > 190 else ''
            if name and (name.startswith(f'({key})') or name == key or name.split(')')[-1].strip() == key):
                return dict(epoch=_unpack_mpc_epoch(line[20:25]), M=float(line[26:35]), Peri=float(line[37:46]),
                            Node=float(line[48:57]), i=float(line[59:68]), e=float(line[70:79]),
                            n=float(line[80:91]), a=float(line[92:103]), line=line.rstrip())
    raise KeyError(f'{target} не найден в {path}')

el = orbital_elements(r_vec, v_vec)
mpc = read_mpcorb(ASTID)
mpc_M = (mpc['M'] + mpc['n']*(JD_EPOCH - mpc['epoch'])) % 360 # перенос M на нашу эпоху
cmp = pd.DataFrame({'Лаплас': el, 'MPCORB': {k: (mpc_M if k == 'M' else mpc[k]) for k in el}})
cmp['разность'] = cmp['Лаплас'] - cmp['MPCORB']
print(f"эпоха JD {JD_EPOCH:.5f}; MPCORB: эпоха элементов JD {mpc['epoch']:.1f}, M перенесена на нашу эпоху")
cmp

эпоха JD 2461313.16080; MPCORB: эпоха элементов JD 2461200.5, M перенесена на нашу эпоху


,Лаплас,MPCORB,разность
a,2.214194,2.210943,0.003251
e,0.546269,0.545700,0.000569
i,23.786108,23.769070,0.017038
Node,203.104898,203.134910,-0.030012
Peri,219.113572,219.122640,-0.009068
M,333.111964,333.051130,0.060834
